In [1]:
# =========================
# BLOCK 1: Install
# =========================
! pip install -q transformers datasets sentence-transformers torch accelerate
# --- NEW: graph dependencies (Colab) ---
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.8 MB/s eta 0:00:00


In [2]:
# =========================
# BLOCK 2: imports
# =========================
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    T5ForConditionalGeneration,
    get_linear_schedule_with_warmup
)

import numpy as np
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

from sentence_transformers import SentenceTransformer
import json
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [3]:
# =========================
# BLOCK 3: set device
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [4]:
# =========================
# BLOCK 4: mount drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# =========================
# BLOCK 5: imports os and set path
# =========================
# import os
# drive_data_path = '/content/drive/MyDrive/LaMP_datasets'
# os.makedirs(drive_data_path, exist_ok=True)
# print(f"Google Drive data path set to: {drive_data_path}")

In [6]:
# =========================
# BLOCK 6: Lamp 3 datasets
# =========================
import os

lamp3_dir = "/content/drive/MyDrive/LaMP_datasets"
#lamp3_dir = "../LaMP_time_3"  # Local path alternative
os.makedirs(lamp3_dir, exist_ok=True)

files = {
    "train_questions.json": "https://ciir.cs.umass.edu/downloads/LaMP/time/LaMP_3/train/train_questions.json",
    "train_outputs.json":   "https://ciir.cs.umass.edu/downloads/LaMP/time/LaMP_3/train/train_outputs.json",
    "dev_questions.json":   "https://ciir.cs.umass.edu/downloads/LaMP/time/LaMP_3/dev/dev_questions.json",
    "dev_outputs.json":     "https://ciir.cs.umass.edu/downloads/LaMP/time/LaMP_3/dev/dev_outputs.json"
}

for fname, url in files.items():
    fpath = os.path.join(lamp3_dir, fname)
    if os.path.exists(fpath):
        print(f"✅ {fname} already exists.")
    else:
        print(f"⬇ Downloading {fname}...")
        os.system(f"wget -q {url} -O {fpath}")


print("\n=== LaMP-3 dataset ready ===")

✅ train_questions.json already exists.
✅ train_outputs.json already exists.
✅ dev_questions.json already exists.
✅ dev_outputs.json already exists.

=== LaMP-3 dataset ready ===


In [7]:
# =============================================
# BLOCK 7: Set configuraions
# model, parameters , A/B switch , configs
# =============================================
class Config:
    # Model parameters
    # llm_name = "google/flan-t5-base"
    # encoder_name = "BAAI/bge-base-en-v1.5"

    # Model parameters local
    #llm_name = "../FlanT5-small"  # Local path alternative
    #llm_name = "/content/drive/MyDrive/FlanT5-small"
    #llm_name = "/content/drive/MyDrive/FlanT5-Large"
    llm_name = "/content/drive/MyDrive/FlanT5-xl"
    #llm_name = "/content/drive/MyDrive/FlanT5-xxl"

    #encoder_name = "../bge-base-en-v1.5"
    encoder_name = "/content/drive/MyDrive/bge-base-en-v1.5"
    #encoder_name = "/content/drive/MyDrive/bge-small-en-v1.5"
    #encoder_name = "BAAI/bge-base-en-v1.5"

    # Training parameters
    batch_size = 8             # Increased for stability
    learning_rate = 2e-4       # Increased from 1e-4 (Need higher LR for new params)
    num_epochs = 10            # Increased from 2
    warmup_ratio = 0.1
    max_input_length = 256
    max_encoder_length = 512
    max_output_length = 128

    # PPlug specific
    embedding_dim = 768
    llm_hidden_size = None  # Will be set in the model initialization
    num_personal_tokens = 1

    # Data parameters
    max_histories = 5          # Reduced for speed
    sample_size = 1000         # for train Increased data size slightly
    test_sample_size = 100     # for test

        # --- NEW: A/B switch ---
   # Supported variants:
    # - "pplug"               -> baseline PPlugModel
    # - "pplug_graph"         -> GraphPPlugModel
    # - "pplug_session"       -> SessionGraphPPlugModel (session only)
    # - "pplug_graph_session" -> SessionGraphPPlugModel
    personalization_variant = "pplug"


    # --- NEW: Graph config ---
class GraphConfig:
    # node feature sizes (we'll keep everything in encoder embedding space initially)
    embedding_dim = 768
    gnn_hidden = 768
    gnn_layers = 3
    review_sim_topk = 3      # build REVIEW_SIM edges between similar history reviews
    review_sim_threshold = 0.35  # cosine similarity threshold


# ==================== session config ====================
class SessionConfig:
    session_len = 3
    session_layers = 2
    session_heads = 4
    session_dropout = 0.1



In [8]:
# =============================================
# BLOCK 8: Load all configs
# =============================================
config = Config()
graph_config = GraphConfig()
session_config = SessionConfig()

In [9]:
# =============================================
# BLOCK 8: data sets , loads and process defs
# =============================================

class LaMP3_Dataset:
    """Loads local LaMP-3 dataset from downloaded JSON files."""
    def __init__(self, split="train", sample_size=500):
        print("LaMP3_Dataset::__init__():Current Working Directory: ", os.getcwd())
        # base_dir = os.getcwd()
        # data_dir = os.path.join(base_dir, "LaMP_time_3")
        #data_dir = "../LaMP_time_3"
        data_dir = "/content/drive/MyDrive/LaMP_datasets"

        file_split = "dev" if split in ["validation", "test"] else "train"
        questions_path = os.path.join(data_dir, f"{file_split}_questions.json")
        outputs_path = os.path.join(data_dir, f"{file_split}_outputs.json")

        if not (os.path.exists(questions_path) and os.path.exists(outputs_path)):
            raise FileNotFoundError("LaMP-3 JSON files not found. Run download first.")

        questions_json = json.load(open(questions_path, "r"))
        outputs_json = json.load(open(outputs_path, "r"))
        gold_dict = {g["id"]: g["output"] for g in outputs_json["golds"]}

        # Truncate for demo
        questions_json = questions_json[:sample_size]
        self.data = self._process_data(questions_json, gold_dict)

    def _process_data(self, questions, gold_dict):
        processed = []
        for q in questions:
            # Histories with score instead of label
            histories = [{"text": h["text"], "label": str(h.get("score", "")),"score": h.get("score", None)}
                         for h in q.get("profile", [])]

            # Prompt for regression
            input_text = q["input"]
            if "rating" not in input_text.lower():
                input_text = f"Predict the product rating (1-5): {input_text}"

            output_text = str(gold_dict.get(q["id"], "0"))
            processed.append({
                "user_id": q["id"],
                "input": input_text,
                "output": output_text,
                "histories": histories
            })
        return processed

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [10]:
# =============================================
# BLOCK 10: User Behavior Encoder
# =============================================
class UserBehaviorEncoder(nn.Module):
    """
    Corrected Encoder using AutoModel directly to allow gradient flow.
    - History Encoder: Frozen
    - Input Encoder: Trainable by default, can be frozen via freeze_input=True
    """
    def __init__(self, encoder_name, freeze_input: bool = False):
        super().__init__()
        self.history_encoder = AutoModel.from_pretrained(encoder_name)
        self.input_encoder = AutoModel.from_pretrained(encoder_name)
        self.tokenizer = AutoTokenizer.from_pretrained(encoder_name)

        self.freeze_input = bool(freeze_input)

        # Freeze History Encoder completely
        for param in self.history_encoder.parameters():
            param.requires_grad = False

        # Optionally freeze Input Encoder
        if self.freeze_input:
            for param in self.input_encoder.parameters():
                param.requires_grad = False

    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def encode_histories(self, texts):
        """Encode histories with NO gradients (Inference mode for efficiency)."""
        device = self.history_encoder.device
        encoded_input = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            model_output = self.history_encoder(**encoded_input)
            embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings

    def encode_input(self, texts):
        """
        Encode input.
        - If input encoder is frozen: no_grad for speed/stability.
        - Else: allow gradients for fine-tuning.
        """
        device = self.input_encoder.device
        encoded_input = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        if self.freeze_input:
            with torch.no_grad():
                model_output = self.input_encoder(**encoded_input)
        else:
            model_output = self.input_encoder(**encoded_input)

        embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings

    def encode_input(self, texts):
        """Encode input WITH gradients iff input_encoder is trainable"""
        device = self.input_encoder.device
        encoded_input = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        model_output = self.input_encoder(**encoded_input)
        embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings

    def encode_input(self, texts):
        """Encode input WITH gradients (Trainable)"""
        device = self.input_encoder.device

        # Tokenize
        encoded_input = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors='pt'
        ).to(device)

        # Forward pass WITH gradients
        model_output = self.input_encoder(**encoded_input)
        embeddings = self.mean_pooling(model_output, encoded_input['attention_mask'])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        return embeddings


In [11]:
# =============================================
# BLOCK 11: Input-aware Personal Aggregator
# =============================================

class PersonalAggregator(nn.Module):
    """Aggregates user histories into personal embedding with attention"""

    def __init__(self, embedding_dim, llm_hidden_size):
        super().__init__()
        # Project from encoder space to LLM space
        self.projector = nn.Sequential(
            nn.Linear(embedding_dim, llm_hidden_size),
            nn.ReLU(),
            nn.Linear(llm_hidden_size, llm_hidden_size)
        )

    def forward(self, history_embeddings, input_embedding):
        """
        Args:
            history_embeddings: [num_histories, embedding_dim]
            input_embedding: [embedding_dim]
        Returns:
            personal_embedding: [llm_hidden_size]
        """
        # Ensure history_embeddings is not an inference tensor and does not require gradients
        history_embeddings_processed = history_embeddings.clone().detach() # FIX: Added .clone()

        # Compute attention weights (Equation 3 in paper)
        # wi = exp(xu^T * hu_i) / sum(exp(xu^T * hu_k))
        scores = torch.matmul(history_embeddings_processed, input_embedding)  # [num_histories]
        weights = torch.softmax(scores, dim=0)  # [num_histories]

        # Weighted aggregation (Equation 4 in paper)
        # Pu = sum(wi * Proj(hu_i))
        projected_histories = self.projector(history_embeddings_processed)  # [num_histories, llm_hidden_size]
        personal_embedding = torch.sum(
            weights.unsqueeze(1) * projected_histories,
            dim=0
        )  # [llm_hidden_size]

        return personal_embedding, weights


In [12]:
# =============================================
# BLOCK 12: PPlug Model
# =============================================

class PPlugModel(nn.Module):
    """Final PPlug Model with corrected Encoder and Masks"""

    def __init__(self, config):
        super().__init__()

        # Load LLM (frozen)
        self.llm = T5ForConditionalGeneration.from_pretrained(config.llm_name)
        self.llm_tokenizer = AutoTokenizer.from_pretrained(config.llm_name)

        for param in self.llm.parameters():
            param.requires_grad = False

        # Infer hidden size from the actual T5 checkpoint (512 for t5-small, 768 for t5-base, ...)
        config.llm_hidden_size = int(self.llm.config.d_model)

        # Corrected User behavior encoder
        self.behavior_encoder = UserBehaviorEncoder(config.encoder_name)

        # Personal aggregator (trainable)
        self.personal_aggregator = PersonalAggregator(
            config.embedding_dim,
            config.llm_hidden_size
        )

        # Small initialization for stability
        self.instruction_embedding = nn.Parameter(
            torch.randn(1, config.num_personal_tokens, config.llm_hidden_size) * 0.01
        )

        self.config = config

    def get_personal_embedding(self, histories, current_input):
        # Encode histories (Frozen path)
        history_texts = [h['text'] for h in histories[:self.config.max_histories]]
        if not history_texts: history_texts = ["Empty"]

        history_embeddings = self.behavior_encoder.encode_histories(history_texts)

        # Encode input (Trainable path)
        # Note: We pass list [current_input] but take [0] index
        input_embedding = self.behavior_encoder.encode_input([current_input])[0]

        # Aggregate
        # Ensure devices match
        target_device = self.instruction_embedding.device
        personal_embedding, attention_weights = self.personal_aggregator(
            history_embeddings.to(target_device),
            input_embedding.to(target_device)
        )

        return personal_embedding, attention_weights

    def forward(self, batch):
        batch_size = len(batch['input'])
        device = self.instruction_embedding.device

        # 1. Tokenize Input & Create Masks
        tokenized_inputs = self.llm_tokenizer(
            batch['input'],
            max_length=self.config.max_input_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        input_ids = tokenized_inputs.input_ids
        original_mask = tokenized_inputs.attention_mask

        # 2. Tokenize Labels
        labels = self.llm_tokenizer(
            batch['output'],
            max_length=self.config.max_output_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).input_ids.to(device)
        labels[labels == self.llm_tokenizer.pad_token_id] = -100

        # 3. Get Embeddings
        inputs_embeds = self.llm.encoder.embed_tokens(input_ids)

        personal_embeds_list = []
        for i in range(batch_size):
            p_emb, _ = self.get_personal_embedding(
                batch['histories'][i],
                batch['input'][i]
            )
            personal_embeds_list.append(p_emb)

        personal_embeds = torch.stack(personal_embeds_list).unsqueeze(1)
        instruction_embeds = self.instruction_embedding.expand(batch_size, -1, -1)

        # 4. Concatenate
        final_embeds = torch.cat([instruction_embeds, personal_embeds, inputs_embeds], dim=1)

        # 5. Fix Attention Mask (Add 1s for the 2 new tokens)
        num_prefix = 1 + self.config.num_personal_tokens
        prefix_mask = torch.ones(batch_size, num_prefix).to(device)
        final_mask = torch.cat([prefix_mask, original_mask], dim=1)

        # 6. Forward
        outputs = self.llm(
            inputs_embeds=final_embeds,
            attention_mask=final_mask,
            labels=labels,
            return_dict=True
        )

        return outputs.loss, outputs.logits

    def generate(self, input_text, histories, max_length=50):
        # ... (Same logic as forward for masking) ...
        device = self.instruction_embedding.device
        personal_emb, attention_weights = self.get_personal_embedding(histories, input_text)

        tokenized = self.llm_tokenizer(
            input_text,
            max_length=self.config.max_input_length,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        input_ids = tokenized.input_ids
        original_mask = tokenized.attention_mask
        inputs_embeds = self.llm.encoder.embed_tokens(input_ids)

        personal_embeds = personal_emb.unsqueeze(0).unsqueeze(0)
        final_embeds = torch.cat([self.instruction_embedding, personal_embeds, inputs_embeds], dim=1)

        num_prefix = 1 + self.config.num_personal_tokens
        prefix_mask = torch.ones(1, num_prefix).to(device)
        final_mask = torch.cat([prefix_mask, original_mask], dim=1)

        with torch.no_grad():
            output_ids = self.llm.generate(
                inputs_embeds=final_embeds,
                attention_mask=final_mask,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )

        return self.llm_tokenizer.decode(output_ids[0], skip_special_tokens=True), attention_weights


In [13]:
# =============================================
# BLOCK 13: data collator
# =============================================

def collate_fn(batch):
    """Custom collate function for DataLoader"""
    return {
        'input': [item['input'] for item in batch],
        #'output': [item['output'] for item in batch],
        'histories': [item['histories'] for item in batch],
        'user_id': [item['user_id'] for item in batch],
        'output': [item['output'] for item in batch]
    }


In [14]:
# =============================================
# BLOCK 14: Training
# =============================================


def train_pplug(model, train_dataset, config):
    """Train PPlug model"""

    # Create dataloader
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )

    # Optimizer (only trainable parameters)
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config.learning_rate
    )

    # Learning rate scheduler
    num_training_steps = len(train_loader) * config.num_epochs
    num_warmup_steps = int(num_training_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
       # Training loop
    model.train()
    global_step = 0

    for epoch in range(config.num_epochs):
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.num_epochs}")

        for batch in progress_bar:
            # Forward pass
            loss, logits = model(batch)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            # Update metrics
            epoch_loss += loss.item()
            global_step += 1

            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg_loss': f'{epoch_loss/global_step:.4f}'
            })

        print(f"\nEpoch {epoch+1} completed. Average loss: {epoch_loss/len(train_loader):.4f}")

    return model



In [15]:
# =============================================
# BLOCK 14: Metrics and Evaluation
# =============================================
def evaluate_pplug(model, test_dataset, num_samples=10):
    """Evaluate PPlug model on test set with robust text matching"""

    model.eval()
    predictions = []
    ground_truths = []

    print("\n" + "="*50)
    print("EVALUATION EXAMPLES")
    print("="*50)

    for i in range(min(num_samples, len(test_dataset))):
        sample = test_dataset[i]

        # Generate prediction
        pred_text, attention_weights = model.generate(
            sample['input'],
            sample['histories'],
            max_length=config.max_output_length
        )

        # CLEANUP: Normalize text for comparison
        pred_clean = pred_text.strip().lower()
        truth_clean = sample['output'].strip().lower()

        predictions.append(pred_clean)
        ground_truths.append(truth_clean)

        # Print examples
        if i < 5:
            print(f"\n--- Example {i+1} ---")
            print(f"Input: {sample['input'][:200]}...")
            print(f"Predicted: '{pred_text}' (Cleaned: '{pred_clean}')")
            print(f"Ground Truth: '{sample['output']}' (Cleaned: '{truth_clean}')")
            print(f"Match: {pred_clean == truth_clean}")

    # Calculate accuracy
    accuracy = accuracy_score(ground_truths, predictions)

    # Convert predictions and ground_truths to numeric for MAE and RMSE
    # Handle cases where conversion might fail (e.g., non-numeric predictions)
    numeric_predictions = []
    numeric_ground_truths = []
    for p, gt in zip(predictions, ground_truths):
        try:
            numeric_predictions.append(float(p))
            numeric_ground_truths.append(float(gt))
        except ValueError:
            # If conversion fails, skip this pair for MAE/RMSE calculation
            continue

    if numeric_predictions and numeric_ground_truths:
        mae = mean_absolute_error(numeric_ground_truths, numeric_predictions)
        rmse = np.sqrt(mean_squared_error(numeric_ground_truths, numeric_predictions))
    else:
        mae = float('nan')
        rmse = float('nan')

    print(f"\n{'='*50}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"{'='*50}")

    return predictions, ground_truths, accuracy, mae, rmse

In [16]:
# =============================================
# BLOCK 15: torch_geometric imports + HeteroGNN
# =============================================
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

class SimpleHeteroGNN(nn.Module):
    """
    Minimal hetero-GNN producing a single pooled 'User' embedding.
    We keep it simple: SAGEConv per relation + mean pooling.
    """
    def __init__(self, hidden_dim, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        self.convs = nn.ModuleList()

        for _ in range(num_layers):
            self.convs.append(
                HeteroConv(
                    {
                        # forward relations
                        ('User', 'RATED', 'Item'): SAGEConv((-1, -1), hidden_dim),
                        ('User', 'WROTE', 'Review'): SAGEConv((-1, -1), hidden_dim),
                        ('Review', 'DESCRIBES', 'Item'): SAGEConv((-1, -1), hidden_dim),
                        ('Review', 'HAS_POLARITY', 'Sentiment'): SAGEConv((-1, -1), hidden_dim),
                        ('Item', 'HAS_POP', 'Popularity'): SAGEConv((-1, -1), hidden_dim),
                        ('Review', 'REVIEW_SIM', 'Review'): SAGEConv((-1, -1), hidden_dim),

                        # reverse relations (CRUCIAL so 'User' receives messages)
                        ('Item', 'RATED_BY', 'User'): SAGEConv((-1, -1), hidden_dim),
                        ('Review', 'WROTE_BY', 'User'): SAGEConv((-1, -1), hidden_dim),
                    },
                    aggr='sum'
                )
            )

    def forward(self, data: HeteroData):
        x_dict = data.x_dict
        edge_index_dict = data.edge_index_dict

        for conv in self.convs:
            x_dict = conv(x_dict, edge_index_dict)
            x_dict = {k: F.relu(v) if v is not None else None for k, v in x_dict.items()}

        user_x = x_dict.get('User', None)
        if user_x is None:
            raise RuntimeError("User node features became None. Check that reverse edges into 'User' exist.")
        return user_x[0]

In [17]:
# =============================================
# BLOCK 16: review node helper
# =============================================

def _topk_review_sim_edges(review_embs: torch.Tensor, topk: int, threshold: float):
    """
    review_embs: [R, D], already normalized.
    returns edge_index [2, E] for ('Review','REVIEW_SIM','Review')
    Always returns a valid LongTensor of shape [2, E] on the same device.
    """
    device = review_embs.device
    R = int(review_embs.size(0))

    # Always return [2, 0] if not enough nodes
    if R <= 1:
        return torch.empty((2, 0), dtype=torch.long, device=device)

    sim = review_embs @ review_embs.t()  # [R,R]
    sim.fill_diagonal_(-1.0)

    edges_src = []
    edges_dst = []

    k = min(int(topk), R - 1)
    for i in range(R):
        vals, idx = torch.topk(sim[i], k=k)
        for v, j in zip(vals.tolist(), idx.tolist()):
            if float(v) >= float(threshold):
                edges_src.append(i)
                edges_dst.append(j)

    if len(edges_src) == 0:
        return torch.empty((2, 0), dtype=torch.long, device=device)

    return torch.tensor([edges_src, edges_dst], dtype=torch.long, device=device)

In [18]:
# =============================================
# BLOCK 17: graph builder
# =============================================


def build_user_sample_graph(
    behavior_encoder: UserBehaviorEncoder,
    histories: list,
    current_input: str,
    embedding_dim: int,
    review_sim_topk: int,
    review_sim_threshold: float,
    device: torch.device
) -> HeteroData:
    data = HeteroData()

    # --- Nodes ---
    data['User'].x = torch.zeros((1, embedding_dim), device=device)

    # Item node feature from current input
    # IMPORTANT: do NOT detach -> keeps graph conditioning input-aware
    item_emb = behavior_encoder.encode_input([current_input])  # [1, D]
    data['Item'].x = item_emb.to(device)

    # Review nodes from histories
    review_texts = [h.get('text', '') for h in histories]
    if len(review_texts) == 0:
        review_texts = ["Empty"]

    review_embs = behavior_encoder.encode_histories(review_texts)  # [R, D]
    data['Review'].x = review_embs.to(device)

    R = int(data['Review'].x.size(0))

    # --- Sentiment nodes (one per review) ---
    sent_feats = []
    for i in range(R):
        rating_val = None
        if i < len(histories):
            rating_val = histories[i].get("score", None)

        if rating_val is None and i < len(histories):
            try:
                rating_val = float(str(histories[i].get("label", "")).strip())
            except Exception:
                rating_val = None

        if rating_val is None:
            polarity = 0.0
        else:
            rating_val = max(1.0, min(5.0, float(rating_val)))
            polarity = (rating_val - 3.0) / 2.0  # [-1, 1]

        # sentiment feature is review embedding scaled by polarity
        sent_feats.append(data["Review"].x[i] * polarity)

    if len(sent_feats) == 0:
        sent_feats = [torch.zeros((embedding_dim,), device=device)]

    data["Sentiment"].x = torch.stack(sent_feats, dim=0).to(device)

    # Popularity node: single node, feature is history length (broadcast across dims)
    pop_scalar = float(len(histories))
    data['Popularity'].x = torch.full((1, embedding_dim), pop_scalar, device=device)

    # --- Edges (ALWAYS long, shape [2, E]) ---
    # User -> Item
    data[('User', 'RATED', 'Item')].edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
    # Item -> User (reverse, so User receives messages)
    data[('Item', 'RATED_BY', 'User')].edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)

    # User -> Review
    data[('User', 'WROTE', 'Review')].edge_index = torch.stack(
        [torch.zeros(R, dtype=torch.long, device=device), torch.arange(R, dtype=torch.long, device=device)],
        dim=0
    )
    # Review -> User (reverse, so User receives messages)
    data[('Review', 'WROTE_BY', 'User')].edge_index = torch.stack(
        [torch.arange(R, dtype=torch.long, device=device), torch.zeros(R, dtype=torch.long, device=device)],
        dim=0
    )

    # Review -> Item
    data[('Review', 'DESCRIBES', 'Item')].edge_index = torch.stack(
        [torch.arange(R, dtype=torch.long, device=device), torch.zeros(R, dtype=torch.long, device=device)],
        dim=0
    )

    # Review -> Sentiment (aligned one-to-one)
    data[('Review', 'HAS_POLARITY', 'Sentiment')].edge_index = torch.stack(
        [torch.arange(R, dtype=torch.long, device=device), torch.arange(R, dtype=torch.long, device=device)],
        dim=0
    )

    # Item -> Popularity
    data[('Item', 'HAS_POP', 'Popularity')].edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)

    # Review similarity edges (may be empty, but must be valid [2, 0])
    sim_edge_index = _topk_review_sim_edges(data['Review'].x, review_sim_topk, review_sim_threshold)
    if sim_edge_index.numel() == 0:
        sim_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
    data[('Review', 'REVIEW_SIM', 'Review')].edge_index = sim_edge_index

    return data

In [19]:
# =============================================
# BLOCK 18: PPlug Model + graph-based personalization
# =============================================


class GraphPPlugModel(nn.Module):
    """Final PPlug Model + optional graph-based personalization"""

    def __init__(self, config, graph_config=None):
        super().__init__()

        # Load LLM (frozen)
        self.llm = T5ForConditionalGeneration.from_pretrained(config.llm_name)
        self.llm_tokenizer = AutoTokenizer.from_pretrained(config.llm_name)
        for param in self.llm.parameters():
            param.requires_grad = False

        # Infer hidden size from the actual T5 checkpoint (512 for t5-small, 768 for t5-base, ...)
        config.llm_hidden_size = int(self.llm.config.d_model)

        # User behavior encoder (history frozen, input trainable)
        self.behavior_encoder = UserBehaviorEncoder(config.encoder_name)

        # PPlug personal aggregator
        self.personal_aggregator = PersonalAggregator(
            config.embedding_dim,
            config.llm_hidden_size
        )

        # Optional graph modules
        self.graph_config = graph_config
        if graph_config is not None:
            self.hetero_gnn = SimpleHeteroGNN(
                hidden_dim=graph_config.gnn_hidden,
                num_layers=graph_config.gnn_layers
            )

            self.graph_to_llm = nn.Sequential(
                nn.Linear(graph_config.gnn_hidden, config.llm_hidden_size),
                nn.ReLU(),
                nn.Linear(config.llm_hidden_size, config.llm_hidden_size),
            )

            # Learnable base feature for the User node (instead of all-zeros)
            self.user_node_emb = nn.Parameter(torch.zeros(graph_config.embedding_dim))
            nn.init.normal_(self.user_node_emb, std=0.02)

            # Gate: decides how much of graph embedding to add (residual)
            self.graph_gate = nn.Sequential(
                nn.Linear(config.llm_hidden_size * 2, 1),
                nn.Sigmoid()
            )
            # Bias gate towards 0 at init (start near baseline PPlug)
            with torch.no_grad():
                self.graph_gate[0].bias.fill_(-2.0)  # sigmoid(-2) ~ 0.12

            # Post-projection after residual fusion (keeps capacity but prevents overwrite)
            self.fuse_post = nn.Sequential(
                nn.Linear(config.llm_hidden_size, config.llm_hidden_size),
                nn.Tanh()
            )
        else:
            self.hetero_gnn = None
            self.graph_to_llm = None
            self.user_node_emb = None
            self.graph_gate = None
            self.fuse_post = None

        # Small initialization for stability
        self.instruction_embedding = nn.Parameter(
            torch.randn(1, config.num_personal_tokens, config.llm_hidden_size) * 0.01
        )

        self.config = config

    def get_personal_embedding(self, histories, current_input):
        """
        Base PPlug personal embedding (long-term) computed the same way as in PPlugModel.
        Returns:
          pplug_emb: [llm_hidden_size]
          attention_weights: [num_histories]
        """
        history_texts = [h['text'] for h in histories[:self.config.max_histories]]
        if not history_texts:
            history_texts = ["Empty"]

        history_embeddings = self.behavior_encoder.encode_histories(history_texts)  # [H, D] (frozen)
        input_embedding = self.behavior_encoder.encode_input([current_input])[0]   # [D] (trainable)

        target_device = self.instruction_embedding.device
        pplug_emb, attention_weights = self.personal_aggregator(
            history_embeddings.to(target_device),
            input_embedding.to(target_device)
        )
        return pplug_emb, attention_weights

    def get_graph_embedding(self, histories, current_input):
        """
        Build a per-sample graph and return user_graph_emb in LLM hidden space.
        NOTE: expects build_user_sample_graph() to exist.
        Recommended: detach Item.x inside build_user_sample_graph() to avoid destabilizing the input encoder.
        """
        if self.graph_config is None:
            return None

        dev = self.instruction_embedding.device
        g = build_user_sample_graph(
            behavior_encoder=self.behavior_encoder,
            histories=histories[:self.config.max_histories],
            current_input=current_input,
            embedding_dim=self.graph_config.embedding_dim,
            review_sim_topk=self.graph_config.review_sim_topk,
            review_sim_threshold=self.graph_config.review_sim_threshold,
            device=dev
        )

        # Inject learnable user node feature
        g['User'].x = self.user_node_emb.unsqueeze(0).to(dev)

        user_graph_emb = self.hetero_gnn(g)            # [gnn_hidden]
        user_graph_emb = self.graph_to_llm(user_graph_emb)  # [llm_hidden]
        return user_graph_emb

    def fuse_pplug_and_graph(self, pplug_emb: torch.Tensor, graph_emb: torch.Tensor):
        """
        Residual gated fusion (safer than concat->MLP):
          fused = pplug + gate(pplug, graph)*graph
        """
        if graph_emb is None:
            return pplug_emb

        gate_inp = torch.cat([pplug_emb, graph_emb], dim=-1)
        g = self.graph_gate(gate_inp)              # [1]
        fused = pplug_emb + g * graph_emb          # [H]
        fused = self.fuse_post(fused)              # [H]
        return fused

    def forward(self, batch):
        batch_size = len(batch['input'])
        device = self.instruction_embedding.device

        tokenized_inputs = self.llm_tokenizer(
            batch['input'],
            max_length=self.config.max_input_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        input_ids = tokenized_inputs.input_ids
        original_mask = tokenized_inputs.attention_mask

        labels = self.llm_tokenizer(
            batch['output'],
            max_length=self.config.max_output_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).input_ids.to(device)
        labels[labels == self.llm_tokenizer.pad_token_id] = -100

        inputs_embeds = self.llm.encoder.embed_tokens(input_ids)

        personal_embeds_list = []
        for i in range(batch_size):
            pplug_emb, _ = self.get_personal_embedding(
                batch['histories'][i],
                batch['input'][i]
            )

            if self.graph_config is not None:
                graph_emb = self.get_graph_embedding(
                    batch['histories'][i],
                    batch['input'][i]
                )
                pplug_emb = self.fuse_pplug_and_graph(pplug_emb, graph_emb)

            personal_embeds_list.append(pplug_emb)

        personal_embeds = torch.stack(personal_embeds_list).unsqueeze(1)
        instruction_embeds = self.instruction_embedding.expand(batch_size, -1, -1)

        final_embeds = torch.cat([instruction_embeds, personal_embeds, inputs_embeds], dim=1)

        num_prefix = 1 + self.config.num_personal_tokens
        prefix_mask = torch.ones(batch_size, num_prefix, device=device)
        final_mask = torch.cat([prefix_mask, original_mask], dim=1)

        outputs = self.llm(
            inputs_embeds=final_embeds,
            attention_mask=final_mask,
            labels=labels,
            return_dict=True
        )

        return outputs.loss, outputs.logits

    def generate(self, input_text, histories, max_length=50):
        device = self.instruction_embedding.device

        # Base PPlug embedding
        personal_emb, attention_weights = self.get_personal_embedding(histories, input_text)

        # Graph fusion (eval matches train)
        if self.graph_config is not None:
            graph_emb = self.get_graph_embedding(histories, input_text)
            personal_emb = self.fuse_pplug_and_graph(personal_emb, graph_emb)

        tokenized = self.llm_tokenizer(
            input_text,
            max_length=self.config.max_input_length,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        input_ids = tokenized.input_ids
        original_mask = tokenized.attention_mask
        inputs_embeds = self.llm.encoder.embed_tokens(input_ids)

        personal_embeds = personal_emb.unsqueeze(0).unsqueeze(0)  # [1,1,H]
        final_embeds = torch.cat([self.instruction_embedding, personal_embeds, inputs_embeds], dim=1)

        num_prefix = int(self.config.num_personal_tokens) + 1
        prefix_mask = torch.ones(1, num_prefix, device=device)
        final_mask = torch.cat([prefix_mask, original_mask], dim=1)

        with torch.no_grad():
            output_ids = self.llm.generate(
                inputs_embeds=final_embeds,
                attention_mask=final_mask,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )

        return self.llm_tokenizer.decode(output_ids[0], skip_special_tokens=True), attention_weights

In [20]:
# ===============================================================
# BLOCK 19: PPlug Model + Session Encoder + Session(+Graph) Model
# ===============================================================
class SessionEncoder(nn.Module):
    """
    Encodes last-N history embeddings (short-term session) into a single vector.
    Input:  [S, D]
    Output: [D]
    """
    def __init__(self, embedding_dim: int, num_layers: int = 2, nhead: int = 4, dropout: float = 0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)

    def forward(self, session_embs: torch.Tensor) -> torch.Tensor:
        # session_embs: [S, D] or [1, S, D]
        if session_embs.dim() == 2:
            session_embs = session_embs.unsqueeze(0)  # [1, S, D]

        out = self.encoder(session_embs)             # [1, S, D]
        return out.mean(dim=1).squeeze(0)            # [D]


class SessionGraphPPlugModel(nn.Module):
    """
    PPlug + Session (last N history items) and optionally + Graph embedding.
    Fusion: residual gated additions onto PPlug embedding (safer than concat fusion).
    """
    def __init__(self, config, session_config, graph_config=None):
        super().__init__()

        # Load LLM (frozen)
        self.llm = T5ForConditionalGeneration.from_pretrained(config.llm_name)
        self.llm_tokenizer = AutoTokenizer.from_pretrained(config.llm_name)
        for param in self.llm.parameters():
            param.requires_grad = False

        # Infer hidden size from the actual T5 checkpoint (512 for t5-small, 768 for t5-base, ...)
        config.llm_hidden_size = int(self.llm.config.d_model)

        # Encoders
        # IMPORTANT: freeze input encoder for stability on small data (prevents training ~100M params)
        self.behavior_encoder = UserBehaviorEncoder(config.encoder_name, freeze_input=True)

        # PPlug personal aggregator
        self.personal_aggregator = PersonalAggregator(config.embedding_dim, config.llm_hidden_size)

        # Session encoder (operates in embedding space first)
        self.session_config = session_config
        self.session_encoder = SessionEncoder(
            embedding_dim=config.embedding_dim,
            num_layers=session_config.session_layers,
            nhead=session_config.session_heads,
            dropout=session_config.session_dropout
        )
        self.session_to_llm = nn.Sequential(
            nn.Linear(config.embedding_dim, config.llm_hidden_size),
            nn.ReLU(),
            nn.Linear(config.llm_hidden_size, config.llm_hidden_size),
        )

        # Optional graph modules
        self.graph_config = graph_config
        if graph_config is not None:
            self.hetero_gnn = SimpleHeteroGNN(
                hidden_dim=graph_config.gnn_hidden,
                num_layers=graph_config.gnn_layers
            )
            self.graph_to_llm = nn.Sequential(
                nn.Linear(graph_config.gnn_hidden, config.llm_hidden_size),
                nn.ReLU(),
                nn.Linear(config.llm_hidden_size, config.llm_hidden_size),
            )

            # Learnable base feature for the User node (instead of all-zeros)
            self.user_node_emb = nn.Parameter(torch.zeros(graph_config.embedding_dim))
            nn.init.normal_(self.user_node_emb, std=0.02)
        else:
            self.hetero_gnn = None
            self.graph_to_llm = None
            self.user_node_emb = None

        # Gates (bias towards not using session/graph early)
        self.session_gate = nn.Sequential(
            nn.Linear(config.llm_hidden_size * 2, 1),
            nn.Sigmoid()
        )
        with torch.no_grad():
            self.session_gate[0].bias.fill_(-2.0)

        if graph_config is not None:
            self.graph_gate = nn.Sequential(
                nn.Linear(config.llm_hidden_size * 2, 1),
                nn.Sigmoid()
            )
            with torch.no_grad():
                self.graph_gate[0].bias.fill_(-2.0)
        else:
            self.graph_gate = None

        # Post projection after residual fusion
        self.fuse_post = nn.Sequential(
            nn.Linear(config.llm_hidden_size, config.llm_hidden_size),
            nn.Tanh()
        )

        # Small initialization for stability
        self.instruction_embedding = nn.Parameter(
            torch.randn(1, config.num_personal_tokens, config.llm_hidden_size) * 0.01
        )

        self.config = config

    def get_personal_embedding(self, histories, current_input):
        # ----- PPlug long-term embedding -----
        history_texts = [h['text'] for h in histories[:self.config.max_histories]]
        if not history_texts:
            history_texts = ["Empty"]

        history_embeddings = self.behavior_encoder.encode_histories(history_texts)  # [H, D] (no grad)
        input_embedding = self.behavior_encoder.encode_input([current_input])[0]   # [D] (frozen via freeze_input=True)

        target_device = self.instruction_embedding.device
        pplug_emb, attention_weights = self.personal_aggregator(
            history_embeddings.to(target_device),
            input_embedding.to(target_device)
        )  # [llm_hidden]

        # ----- Session embedding from last N histories -----
        session_len = min(self.session_config.session_len, int(history_embeddings.size(0)))
        session_slice = history_embeddings[-session_len:]  # [S, D]
        session_vec = self.session_encoder(session_slice.to(target_device))  # [D]
        session_vec = self.session_to_llm(session_vec)  # [llm_hidden]

        fused = pplug_emb

        # Gate session into pplug (residual)
        sg = self.session_gate(torch.cat([pplug_emb, session_vec], dim=-1))  # [1]
        fused = fused + sg * session_vec

        # ----- Optional graph embedding -----
        if self.graph_config is not None:
            g = build_user_sample_graph(
                behavior_encoder=self.behavior_encoder,
                histories=histories[:self.config.max_histories],
                current_input=current_input,
                embedding_dim=self.graph_config.embedding_dim,
                review_sim_topk=self.graph_config.review_sim_topk,
                review_sim_threshold=self.graph_config.review_sim_threshold,
                device=target_device
            )

            # Inject learnable user node feature
            g['User'].x = self.user_node_emb.unsqueeze(0).to(target_device)

            user_graph_emb = self.hetero_gnn(g)                 # [gnn_hidden]
            user_graph_emb = self.graph_to_llm(user_graph_emb)  # [llm_hidden]

            gg = self.graph_gate(torch.cat([fused, user_graph_emb], dim=-1))  # [1]
            fused = fused + gg * user_graph_emb

        fused = self.fuse_post(fused)
        return fused, attention_weights

    def forward(self, batch):
        batch_size = len(batch['input'])
        device = self.instruction_embedding.device

        tokenized_inputs = self.llm_tokenizer(
            batch['input'],
            max_length=self.config.max_input_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        input_ids = tokenized_inputs.input_ids
        original_mask = tokenized_inputs.attention_mask

        labels = self.llm_tokenizer(
            batch['output'],
            max_length=self.config.max_output_length,
            padding=True,
            truncation=True,
            return_tensors='pt'
        ).input_ids.to(device)
        labels[labels == self.llm_tokenizer.pad_token_id] = -100

        inputs_embeds = self.llm.encoder.embed_tokens(input_ids)

        personal_embeds_list = []
        for i in range(batch_size):
            p_emb, _ = self.get_personal_embedding(batch['histories'][i], batch['input'][i])
            personal_embeds_list.append(p_emb)

        personal_embeds = torch.stack(personal_embeds_list).unsqueeze(1)  # [B,1,H]
        instruction_embeds = self.instruction_embedding.expand(batch_size, -1, -1)

        final_embeds = torch.cat([instruction_embeds, personal_embeds, inputs_embeds], dim=1)

        num_prefix = 1 + self.config.num_personal_tokens
        prefix_mask = torch.ones(batch_size, num_prefix, device=device)
        final_mask = torch.cat([prefix_mask, original_mask], dim=1)

        outputs = self.llm(
            inputs_embeds=final_embeds,
            attention_mask=final_mask,
            labels=labels,
            return_dict=True
        )
        return outputs.loss, outputs.logits

    def generate(self, input_text, histories, max_length=50):
        device = self.instruction_embedding.device

        personal_emb, attention_weights = self.get_personal_embedding(histories, input_text)

        tokenized = self.llm_tokenizer(
            input_text,
            max_length=self.config.max_input_length,
            truncation=True,
            return_tensors='pt'
        ).to(device)

        input_ids = tokenized.input_ids
        original_mask = tokenized.attention_mask
        inputs_embeds = self.llm.encoder.embed_tokens(input_ids)

        personal_embeds = personal_emb.unsqueeze(0).unsqueeze(0)  # [1,1,H]
        final_embeds = torch.cat([self.instruction_embedding, personal_embeds, inputs_embeds], dim=1)

        num_prefix = 1 + self.config.num_personal_tokens
        prefix_mask = torch.ones(1, num_prefix, device=device)
        final_mask = torch.cat([prefix_mask, original_mask], dim=1)

        with torch.no_grad():
            output_ids = self.llm.generate(
                inputs_embeds=final_embeds,
                attention_mask=final_mask,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )

        return self.llm_tokenizer.decode(output_ids[0], skip_special_tokens=True), attention_weights

In [21]:
# ===============================================================
# BLOCK 19: Checkpointing & Loading
# ===============================================================

def save_checkpoint(model, config, variant, out_dir, metrics=None):
    os.makedirs(out_dir, exist_ok=True)
    ckpt_path = os.path.join(out_dir, "model.pt")
    meta_path = os.path.join(out_dir, "meta.json")

    # Save only trainable weights + everything else in model state (safe/simple)
    torch.save(model.state_dict(), ckpt_path)

    meta = {
        "variant": variant,
        "metrics": metrics,
        "config": {
            "llm_name": config.llm_name,
            "encoder_name": config.encoder_name,
            "batch_size": config.batch_size,
            "learning_rate": config.learning_rate,
            "num_epochs": config.num_epochs,
            "max_histories": config.max_histories,
            "max_input_length": config.max_input_length,
            "max_output_length": config.max_output_length,
            "embedding_dim": config.embedding_dim,
            "llm_hidden_size": config.llm_hidden_size,
            "num_personal_tokens": config.num_personal_tokens,
        }
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    return ckpt_path, meta_path


def load_model_for_inference(variant, config, session_config, graph_config, ckpt_path, device):
    use_graph = (variant in ["pplug_graph", "pplug_graph_session"])
    use_session = (variant in ["pplug_session", "pplug_graph_session"])

    if use_session:
        model = SessionGraphPPlugModel(
            config,
            session_config=session_config,
            graph_config=(graph_config if use_graph else None)
        ).to(device)
    elif use_graph:
        model = GraphPPlugModel(config, graph_config=graph_config).to(device)
    else:
        model = PPlugModel(config).to(device)

    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

In [22]:
# ===============================================================
# BLOCK 20: offline inference
# ===============================================================
# variant = "pplug_graph_session"
# ckpt_path = "./logs/checkpoints/<run_id>/pplug_graph_session/model.pt"

# model = load_model_for_inference(
#     variant=variant,
#     config=config,
#     session_config=session_config,
#     graph_config=graph_config,
#     ckpt_path=ckpt_path,
#     device=device
# )

# sample = LaMP3_Dataset(split="validation", sample_size=1)[0]
# pred, _ = model.generate(sample["input"], sample["histories"], max_length=config.max_output_length)
# print(pred)

In [23]:
# ===============================================================
# BLOCK 21: Main Entry
# ===============================================================
def main():
    print("="*60)
    print("PPlug: Personalized LLM Implementation")
    print("Based on: LLMs + Persona-Plug = Personalized LLMs")
    print("="*60)

    print("\n1. Loading datasets...")
    train_dataset = LaMP3_Dataset(split="train", sample_size=config.sample_size)
    test_dataset = LaMP3_Dataset(split="validation", sample_size=config.test_sample_size)
    print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

    # Supported variants:
    # - "pplug"               -> baseline PPlugModel
    # - "pplug_graph"         -> GraphPPlugModel
    # - "pplug_session"       -> SessionGraphPPlugModel (session only)
    # - "pplug_graph_session" -> SessionGraphPPlugModel (graph + session)
    variant = getattr(config, "personalization_variant", "pplug_graph")

    use_graph = (variant in ["pplug_graph", "pplug_graph_session"])
    use_session = (variant in ["pplug_session", "pplug_graph_session"])

    print("\n2. Initializing model...")
    if use_session:
        model = SessionGraphPPlugModel(
            config,
            session_config=session_config,
            graph_config=(graph_config if use_graph else None)
        ).to(device)
    elif use_graph:
        model = GraphPPlugModel(config, graph_config=graph_config).to(device)
    else:
        model = PPlugModel(config).to(device)

    print(f"Variant: {variant} | Model: {model.__class__.__name__}")

    # Warmup: only needed for graph variants (lazy torch_geometric init)
    if use_graph:
        _warmup_sample = train_dataset[0]
        _warmup_batch = {
            "input": [_warmup_sample["input"]],
            "output": [_warmup_sample["output"]],
            "histories": [_warmup_sample["histories"]],
            "user_id": [_warmup_sample["user_id"]],
        }
        with torch.no_grad():
                _ = model(_warmup_batch)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")

    print("\n3. Training...")
    model = train_pplug(model, train_dataset, config)

    print("\n4. Evaluating...")
    predictions, ground_truths, accuracy, mae, rmse = evaluate_pplug(model, test_dataset, num_samples=len(test_dataset))

    print("\n" + "="*60)
    print("Training and Evaluation Complete!")
    print("="*60)

    return model, predictions, ground_truths, accuracy, mae, rmse

In [24]:
# ===============================================================
# BLOCK 22: Stand alone Runner
# ===============================================================
# model, predictions, ground_truths, accuracy, mae, rmse = main()

In [25]:
# ===============================================================
# BLOCK 23: Multivaritent runner
# ===============================================================
import os
import json
import time
import traceback
import sys
from datetime import datetime

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

    def isatty(self):
        return False

def run_variants_and_log(variants, logs_dir="./logs", also_print_to_console=True):
    print("run_variants_and_log():Current Working Directory: ", os.getcwd())
    os.makedirs(logs_dir, exist_ok=True)
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    summary_path = os.path.join(logs_dir, f"summary_{run_id}.jsonl")

    results = []

    for variant in variants:
        config.personalization_variant = variant

        log_path = os.path.join(logs_dir, f"{run_id}_{variant}.log")
        meta = {
            "run_id": run_id,
            "variant": variant,
            "start_time": datetime.now().isoformat(),
            "log_path": os.path.abspath(log_path),
            "config": {
                "sample_size": config.sample_size,
                "test_sample_size": config.test_sample_size,
                "batch_size": config.batch_size,
                "learning_rate": config.learning_rate,
                "num_epochs": config.num_epochs,
                "max_histories": config.max_histories,
            }
        }

        t0 = time.time()
        error = None
        metrics = None

        try:
            with open(log_path, "w") as lf:
                old_stdout = sys.stdout
                sys.stdout = Tee(old_stdout, lf) if also_print_to_console else lf
                try:
                    print(f"=== RUN START: {variant} ===")
                    print(json.dumps(meta, indent=2))
                    model, predictions, ground_truths, accuracy, mae, rmse = main()
                    metrics = {"accuracy": float(accuracy), "mae": float(mae), "rmse": float(rmse)}

                    # ckpt_dir = os.path.join(logs_dir, "checkpoints", run_id, variant)
                    # ckpt_path, meta_path = save_checkpoint(model, config, variant, ckpt_dir, metrics=metrics)
                    # print(f"\n=== CHECKPOINT SAVED ===\nckpt: {ckpt_path}\nmeta: {meta_path}\n")

                    print("\n=== METRICS ===")
                    print(json.dumps(metrics, indent=2))
                    print(f"=== RUN END: {variant} ===")
                finally:
                    sys.stdout = old_stdout

        except Exception as e:
            error = {
                "type": type(e).__name__,
                "message": str(e),
                "traceback": traceback.format_exc(),
            }
            with open(log_path, "a") as lf:
                lf.write("\n\n=== RUN ERROR ===\n")
                lf.write(json.dumps(error, indent=2))
                lf.write("\n")

        elapsed = time.time() - t0
        record = {**meta, "end_time": datetime.now().isoformat(), "elapsed_sec": elapsed, "metrics": metrics, "error": error}
        results.append(record)

        with open(summary_path, "a") as sf:
            sf.write(json.dumps(record) + "\n")

    return results

variants_to_run = ["pplug", "pplug_graph", "pplug_graph_session"]
results = run_variants_and_log(variants_to_run, logs_dir="/content/drive/MyDrive/logs", also_print_to_console=True)

run_variants_and_log():Current Working Directory:  /content
=== RUN START: pplug ===
{
  "run_id": "20260204_220636",
  "variant": "pplug",
  "start_time": "2026-02-04T22:06:36.444080",
  "log_path": "/content/drive/MyDrive/logs/20260204_220636_pplug.log",
  "config": {
    "sample_size": 1000,
    "test_sample_size": 100,
    "batch_size": 8,
    "learning_rate": 0.0002,
    "num_epochs": 10,
    "max_histories": 5
  }
}
PPlug: Personalized LLM Implementation
Based on: LLMs + Persona-Plug = Personalized LLMs

1. Loading datasets...
LaMP3_Dataset::__init__():Current Working Directory:  /content
LaMP3_Dataset::__init__():Current Working Directory:  /content
Train size: 1000, Test size: 100

2. Initializing model...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /content/drive/MyDrive/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...


Epoch 8 completed. Average loss: 9.8042


Epoch 9/10:   0%|          | 0/125 [00:00<?, ?it/s]


Epoch 9 completed. Average loss: 9.8029


Epoch 10/10:   0%|          | 0/125 [00:00<?, ?it/s]


Epoch 10 completed. Average loss: 9.8014

4. Evaluating...

EVALUATION EXAMPLES

--- Example 1 ---
Input: Predict the product rating (1-5): What is the score of the following review on a scale of 1 to 5? just answer with 1, 2, 3, 4, or 5 without further explanation. review: If You Were Me and Lived In...G...
Predicted: '5' (Cleaned: '5')
Ground Truth: '5' (Cleaned: '5')
Match: True

--- Example 2 ---
Input: Predict the product rating (1-5): What is the score of the following review on a scale of 1 to 5? just answer with 1, 2, 3, 4, or 5 without further explanation. review: This is more than a cookbook; i...
Predicted: '5' (Cleaned: '5')
Ground Truth: '5' (Cleaned: '5')
Match: True

--- Example 3 ---
Input: Predict the product rating (1-5): What is the score of the following review on a scale of 1 to 5? just answer with 1, 2, 3, 4, or 5 without further explanation. review: I LOVE how she evolved from a l...
Predicted: '5' (Cleaned: '5')
Ground Truth: '5' (Cleaned: '5')
Match: True

---